In [6]:
import pandas as pd

In [7]:
y_path = "/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/feature_repo/data/labels.parquet"
y = pd.read_parquet(y_path)

y.head()

,user_id,event_timestamp,churned
0,717,2024-04-01,0
1,266,2024-04-01,0
2,730,2024-04-01,0
3,87,2024-04-01,0
4,374,2024-04-01,0


In [8]:
demographic_path = "/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/feature_repo/data/user_demographics.parquet"
demographic = pd.read_parquet(demographic_path)
demographic.head()

,user_id,age,country,signup_date,event_timestamp,created
0,1,56,US,2023-12-14,2024-01-01,2024-01-01
1,2,69,VN,2023-04-25,2024-01-01,2024-01-01
2,3,46,VN,2023-04-14,2024-01-01,2024-01-01
3,4,32,VN,2023-04-28,2024-01-01,2024-01-01
4,5,60,VN,2023-01-05,2024-01-01,2024-01-01


In [9]:
transaction_path = "/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/feature_repo/data/user_transactions.parquet"
transaction = pd.read_parquet(transaction_path)

transaction.head()


,user_id,event_timestamp,avg_purchase_7d,total_orders_30d,days_since_last_purchase,created
0,1,2024-01-01 00:00:00.000000000,366.88,1,29,2024-01-01 00:00:00.000000000
1,1,2024-01-17 10:54:32.727272727,250.74,5,24,2024-01-17 10:54:32.727272727
2,1,2024-02-02 21:49:05.454545454,402.90,6,1,2024-02-02 21:49:05.454545454
3,1,2024-02-19 08:43:38.181818182,206.75,5,26,2024-02-19 08:43:38.181818182
4,1,2024-03-06 19:38:10.909090909,335.97,3,23,2024-03-06 19:38:10.909090909


In [10]:
transaction.event_timestamp.min()

Timestamp('2024-01-01 00:00:00')

In [11]:
transaction.event_timestamp.max()

Timestamp('2024-06-30 00:00:00')

# Phase 2: Training với Historical Features (Point-in-Time Join)

Đây là phần quan trọng nhất của Feast - lấy feature ĐÚNG tại thời điểm cần để train không bị data leakage.



In [12]:
import pandas as pd
from feast import FeatureStore
from pathlib import Path

REPO_PATH = Path("/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/feature_repo")
store = FeatureStore(repo_path=str(REPO_PATH))

# Xem raw transaction của user_id = 1

print("=" * 60)
print("RAW DATA: Transactions cua user_id=1")
print("=" * 60)
tx_df = pd.read_parquet(REPO_PATH / "data" / "user_transactions.parquet")
user1_tx = tx_df[tx_df["user_id"] == 1].sort_values("event_timestamp")
user1_tx[["event_timestamp", "avg_purchase_7d", "total_orders_30d"]].head(10)

RAW DATA: Transactions cua user_id=1


,event_timestamp,avg_purchase_7d,total_orders_30d
0,2024-01-01 00:00:00.000000000,366.88,1
1,2024-01-17 10:54:32.727272727,250.74,5
2,2024-02-02 21:49:05.454545454,402.90,6
3,2024-02-19 08:43:38.181818182,206.75,5
4,2024-03-06 19:38:10.909090909,335.97,3
5,2024-03-23 06:32:43.636363636,426.52,4
6,2024-04-08 17:27:16.363636364,37.54,7
7,2024-04-25 04:21:49.090909090,112.13,4
8,2024-05-11 15:16:21.818181818,194.93,6
9,2024-05-28 02:10:54.545454546,321.56,3


In [13]:
store = FeatureStore(repo_path=str(REPO_PATH))
print(store)

FeatureStore(
    repo_path=PosixPath('/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/feature_repo'),
    config=RepoConfig(project='churn_prediction', project_description=None, provider='local', registry_config='data/registry.db', online_config={'type': 'sqlite', 'path': 'data/online_store.db'}, auth={'type': 'no_auth'}, offline_config={'type': 'file'}, batch_engine_config='local', feature_server=None, flags=None, repo_path=PosixPath('/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/feature_repo'), entity_key_serialization_version=3, coerce_tz_aware=True, materialization_config=MaterializationConfig(pull_latest_features=False, online_write_batch_size=None), openlineage_config=None),
    registry=not loaded,
    provider=not loaded
)


In [14]:
print("\n" + "=" * 60)
print("QUERY: Lay feature cua user_id=1 tai 3 thoi diem")
print("=" * 60)
 
query_df = pd.DataFrame({
    "user_id": [1, 1, 1],
    "event_timestamp": pd.to_datetime([
        "2024-02-01",  # Som
        "2024-04-15",  # Giua
        "2024-06-20",  # Muon
    ]),
})


QUERY: Lay feature cua user_id=1 tai 3 thoi diem


In [15]:
result = store.get_historical_features(
    entity_df=query_df,
    features=[
        "user_transaction_stats:avg_purchase_7d",
        "user_transaction_stats:total_orders_30d",
    ],
).to_df()

result


,user_id,event_timestamp,avg_purchase_7d,total_orders_30d
0,1,2024-02-01 00:00:00+00:00,250.74,5
1,1,2024-04-15 00:00:00+00:00,37.54,7
2,1,2024-06-20 00:00:00+00:00,388.32,4


In [16]:
print(result)
 
print("\n" + "=" * 60)
print("KET LUAN")
print("=" * 60)
print("""
Voi moi query timestamp, Feast tu dong:
1. Tìm mỗi row có user_id = 1 và timestamp <= Query timestamp
2. Lấy row mới nhất = "Latest"
3. Tự loại bỏ nếu vượt quá TTL (Time-to-live)
 
-> Không bao giờ bị data leakage và không bị dùng feature quá cũ

""")

   user_id           event_timestamp  avg_purchase_7d  total_orders_30d
0        1 2024-02-01 00:00:00+00:00           250.74                 5
1        1 2024-04-15 00:00:00+00:00            37.54                 7
2        1 2024-06-20 00:00:00+00:00           388.32                 4

KET LUAN

Voi moi query timestamp, Feast tu dong:
1. Tìm mỗi row có user_id = 1 và timestamp <= Query timestamp
2. Lấy row mới nhất = "Latest"
3. Tự loại bỏ nếu vượt quá TTL (Time-to-live)

-> Không bao giờ bị data leakage và không bị dùng feature quá cũ




# Phase 3: Serving Real-Time với Materialization

### 1. Materialization = Materialize Features: là quá trình vận chuyển và đồng bộ dữ liệu đặc trưng (features) từ kho lưu trữ (Offline Store) sang Online Store


In [27]:
from datetime import datetime

# 3.1: Materialize Features

store = FeatureStore(repo_path=str(REPO_PATH))

# Chạy materialize từ quá khứ đến thời điểm hiện tại

store.materialize(
    start_date=datetime(2024,1,1),
    end_date = datetime(2024,7,1),
)

Materializing 2 feature views from 2024-01-01 00:00:00+00:00 to 2024-07-01 00:00:00+00:00 into the sqlite online store.

user_demographics:
user_transaction_stats:


In [ ]:
# %cd /home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/feature_repo
# ! feast apply

/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/feature_repo


/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=102131) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


No project found in the repository. Using project name churn_prediction defined in feature_store.yaml
Applying changes for project churn_prediction
/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/.venv/lib/python3.12/site-packages/feast/feature_store.py:690: RuntimeWarning: On demand feature view is an experimental feature. This API is stable, but the functionality does not scale well for offline retrieval
  warnings.warn(
Updated feature view user_demographics
	batch_source: type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created"
file_options {
  uri: "/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/feature_repo/data/user_demographics.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "user_demographic_source"
meta {
  created_timestamp {
    seconds: 1778987108
    nanos: 335402000
  }
  last_updated_timestamp {
    seconds: 1778987108
    nanos: 335402000
  }
}
 -> type:

In [38]:
# 3.2: Serve Real-time

"""
Real-time serving với Online Store

Khác với get_historical_features (chậm, đọc từ Parquet)

- get_online_features đọc từ Online Store (SQLite/Redis)
- Latency < 10ms
- Chỉ trả về giá trị MỚI NHẤT (khong co lich su)
"""
import time
import pickle
import joblib
import pandas as pd
import numpy as np
from feast import FeatureStore
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier

In [41]:
!pip install scikit-learn

/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=102131) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


In [43]:
import joblib  # Thay vì pickle
from pathlib import Path
from feast import FeatureStore
from sklearn.ensemble import RandomForestClassifier

store = FeatureStore(repo_path=str(REPO_PATH))

# Load model bằng joblib (tốt hơn pickle cho sklearn)
model_path = Path("/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/scripts/model.pkl")
model_data = joblib.load(model_path)

model = model_data["model"]
feature_cols = model_data["feature_cols"]

print("✅ Đã load model thành công!")
print("Danh sách features:", feature_cols)
print(f"Model type: {type(model).__name__}")

✅ Đã load model thành công!
Danh sách features: ['age', 'avg_purchase_7d', 'total_orders_30d', 'days_since_last_purchase', 'country_JP', 'country_SG', 'country_US', 'country_VN']
Model type: RandomForestClassifier


In [46]:
print("=" * 60)
print("STEP 1: Get online features cho 5 user")
print("=" * 60)
 
start = time.time()
features = store.get_online_features(
    features=[
        "user_demographics:age",
        "user_demographics:country",
        "user_transaction_stats:avg_purchase_7d",
        "user_transaction_stats:total_orders_30d",
        "user_transaction_stats:days_since_last_purchase",
    ],
    entity_rows=[
        {"user_id": 1},
        {"user_id": 2},
        {"user_id": 3},
        {"user_id": 100},
        {"user_id": 500},
    ],
).to_dict()
latency_ms = (time.time() - start) * 1000

print(f"Latency: {latency_ms:.2f}ms")
print(f"Features returned: {features}")

print("\n" + "=" * 60)
print("STEP 2: Predict tu features")
print("=" * 60)

STEP 1: Get online features cho 5 user
Latency: 5.21ms
Features returned: {'user_id': [1, 2, 3, 100, 500], 'country': ['US', 'VN', 'VN', 'SG', 'VN'], 'age': [56, 69, 46, 32, 18], 'avg_purchase_7d': [142.52000427246094, 386.29998779296875, 25.510000228881836, 76.19999694824219, 86.69000244140625], 'total_orders_30d': [2, 4, 3, 2, 4], 'days_since_last_purchase': [11, 4, 12, 7, 15]}

STEP 2: Predict tu features


In [49]:
print("\n" + "=" * 60)
print("STEP 2: Predict tu features")
print("=" * 60)
# Chuyen thanh DataFrame
df = pd.DataFrame(features)
print(f"Raw features:\n{df}")
 
# Tien xu ly: one-hot encode country (giong luc train)
df = pd.get_dummies(df, columns=["country"], prefix="country")
 
# Dam bao co du cot (neu user khong co country JP, can them cot = 0)
for col in feature_cols:
    if col not in df.columns:
        df[col] = 0
 
# Sap xep cot theo dung thu tu luc train
X = df[feature_cols]
print(f"\nFeatures for prediction:\n{X}")
 
# Predict
predictions = model.predict_proba(X)[:, 1]
print(f"\nChurn probabilities: {predictions}")
 
for uid, prob in zip(features["user_id"], predictions):
    risk = "HIGH" if prob > 0.5 else "LOW"
    print(f"  User {uid}: churn_prob={prob:.3f} [{risk}]")
 
print("\n" + "=" * 60)
print("BENCHMARK: 100 requests")
print("=" * 60)
 
latencies = []
for _ in range(100):
    start = time.time()
    store.get_online_features(
        features=[
            "user_transaction_stats:avg_purchase_7d",
            "user_transaction_stats:total_orders_30d",
        ],
        entity_rows=[{"user_id": int(np.random.randint(1, 1000))}],
    ).to_dict()
    latencies.append((time.time() - start) * 1000)
 
print(f"P50 latency: {np.percentile(latencies, 50):.2f}ms")
print(f"P95 latency: {np.percentile(latencies, 95):.2f}ms")
print(f"P99 latency: {np.percentile(latencies, 99):.2f}ms")


STEP 2: Predict tu features
Raw features:
   user_id country  age  avg_purchase_7d  total_orders_30d  \
0        1      US   56       142.520004                 2   
1        2      VN   69       386.299988                 4   
2        3      VN   46        25.510000                 3   
3      100      SG   32        76.199997                 2   
4      500      VN   18        86.690002                 4   

   days_since_last_purchase  
0                        11  
1                         4  
2                        12  
3                         7  
4                        15  

Features for prediction:
   age  avg_purchase_7d  total_orders_30d  days_since_last_purchase  \
0   56       142.520004                 2                        11   
1   69       386.299988                 4                         4   
2   46        25.510000                 3                        12   
3   32        76.199997                 2                         7   
4   18        86.690002

## Phase 4: Production Setup - FastAPI + Online Store



In [ ]:
# Tạo FastAPI Service

"""
FastAPI inference service tich hop voi Feast Online Store.

Production-ready features:
- Stateless: khong luu trang thai trong RAM
- Health check endpoint
- Async support
- Proper error handling
"""
from contextlib import asynccontextmanager
from pathlib import Path
from typing import List

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from feast import FeatureStore
from pydantic import BaseModel

# ============================================================
# CONFIG
# ============================================================
REPO_PATH =  Path("/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/feature_repo")
MODEL_PATH = Path("/home/trunghieu/projects/my-feast-project/Customer-Churn-Prediction/scripts/model.pkl")

# Globals duoc load 1 lan luc startup - stateless cho moi request
state = {}


# ============================================================
# LIFESPAN: Load model + feast store khi startup
# ============================================================
@asynccontextmanager
async def lifespan(app: FastAPI):
    print("[STARTUP] Loading Feast store...")
    state["store"] = FeatureStore(repo_path=str(REPO_PATH))
    
    print("[STARTUP] Loading model...")
    model_data = joblib.load(MODEL_PATH)
    state["model"] = model_data["model"]
    state["feature_cols"] = model_data["feature_cols"]
    
    print("[STARTUP] Ready!")
    yield
    print("[SHUTDOWN] Cleaning up...")
    state.clear()


app = FastAPI(
    title="Churn Prediction API",
    version="1.0.0",
    lifespan=lifespan,
)


# ============================================================
# REQUEST/RESPONSE SCHEMAS
# ============================================================
class PredictRequest(BaseModel):
    user_ids: List[int]


class PredictResponse(BaseModel):
    user_id: int
    churn_probability: float
    risk_level: str
    features_used: dict


# ============================================================
# ENDPOINTS
# ============================================================
@app.get("/health")
async def health():
    """Health check - K8s dung de biet pod san sang."""
    return {
        "status": "ok",
        "model_loaded": "model" in state,
        "store_loaded": "store" in state,
    }


@app.post("/predict", response_model=List[PredictResponse])
async def predict(request: PredictRequest):
    """
    Predict churn probability cho danh sach user.
    
    Workflow:
    1. Get online features tu Feast
    2. Preprocess
    3. Model predict
    4. Return
    """
    try:
        store: FeatureStore = state["store"]
        model = state["model"]
        feature_cols = state["feature_cols"]
        
        # Step 1: Fetch features tu online store
        entity_rows = [{"user_id": uid} for uid in request.user_ids]
        features = store.get_online_features(
            features=[
                "user_demographics:age",
                "user_demographics:country",
                "user_transaction_stats:avg_purchase_7d",
                "user_transaction_stats:total_orders_30d",
                "user_transaction_stats:days_since_last_purchase",
            ],
            entity_rows=entity_rows,
        ).to_dict()
        
        # Step 2: Preprocess
        df = pd.DataFrame(features)
        
        # Validate: kiem tra co user nao bi missing feature khong
        if df.isnull().any().any():
            missing_users = df[df.isnull().any(axis=1)]["user_id"].tolist()
            raise HTTPException(
                status_code=404,
                detail=f"Features missing for users: {missing_users}",
            )
        
        # One-hot encode country
        df_features = pd.get_dummies(df, columns=["country"], prefix="country")
        for col in feature_cols:
            if col not in df_features.columns:
                df_features[col] = 0
        X = df_features[feature_cols]
        
        # Step 3: Predict
        probabilities = model.predict_proba(X)[:, 1]
        
        # Step 4: Build response
        results = []
        for i, uid in enumerate(request.user_ids):
            prob = float(probabilities[i])
            results.append(PredictResponse(
                user_id=uid,
                churn_probability=round(prob, 4),
                risk_level="HIGH" if prob > 0.5 else "LOW",
                features_used={
                    "age": int(features["age"][i]),
                    "country": features["country"][i],
                    "avg_purchase_7d": round(float(features["avg_purchase_7d"][i]), 2),
                    "total_orders_30d": int(features["total_orders_30d"][i]),
                },
            ))
        
        return results
    
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/feature-views")
async def list_feature_views():
    """List all feature views in registry - debug endpoint."""
    store = state["store"]
    fvs = store.list_feature_views()
    return [
        {
            "name": fv.name,
            "entities": [e for e in fv.entities],
            "features": [f.name for f in fv.features],
            "ttl_days": fv.ttl.days if fv.ttl else None,
        }
        for fv in fvs
    ]